# Assessment 2: Machine Learning and Real-Time Streaming
**Student ID:** 33602190  
**Unit:** ITO5202 · **Year:** 2026

This notebook will use the approved IBM AML HI-Small dataset from Assessment 1. Part A will compare classification pipelines; Part B will apply the saved pipeline to held-out transactions replayed through Kafka.

## 0. Environment and configuration
Start Kafka with `bash scripts/start_kafka.sh` in a separate terminal, then open the notebook using `bash scripts/start_notebook.sh`. Select **Python (assessment2-aml)** and run from the top.

Configuration: project-relative paths, random seed 42 and UTC timestamps. The first Spark start downloads the Kafka connector and its dependencies.

In [1]:
from pathlib import Path
import os
import sys
import platform
from importlib.metadata import version

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / "requirements.txt").exists(), "Use the repo root."
JAVA_HOME = PROJECT_ROOT / ".runtime/jdk-17.0.20.1+1/Contents/Home"
assert (JAVA_HOME / "bin/java").exists(), "Install Java; see README."
expected_environment = (PROJECT_ROOT / ".venv").resolve()
assert Path(sys.prefix).resolve() == expected_environment, "Use the A2 kernel."
os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 4g pyspark-shell"

SEED = 42
DATA_DIR = PROJECT_ROOT / "data"
MODEL_PATH = PROJECT_ROOT / "models/a2_model"
STREAM_PATH = DATA_DIR / "stream_holdout"
KAFKA_BOOTSTRAP_SERVERS = "127.0.0.1:9092"
KAFKA_TOPIC = "aml-transactions"
RUN_KAFKA_SMOKE_TEST = True
SPARK_KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7"

assert platform.python_version() == "3.11.17"
assert version("pyspark") == "3.5.7"
print("Python:", platform.python_version())
print("PySpark:", version("pyspark"))
print("Kafka Python client:", version("kafka-python3"))
print("Kernel: assessment2-aml")

Python: 3.11.17
PySpark: 3.5.7
Kafka Python client: 3.0.0
Kernel: assessment2-aml


### Start Spark
Use four local worker threads and a 4 GB driver. The connector version matches Spark 3.5.7 and its Scala 2.12 build. Kafka's server uses Scala 2.13 independently; the Spark connector must match Spark's Scala version.

In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[4]")
         .appName("Assessment2-AML")
         .config("spark.jars.packages", SPARK_KAFKA_PACKAGE)
         .config("spark.jars.ivy", str(PROJECT_ROOT / ".runtime/ivy"))
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
jvm = spark.sparkContext._jvm
java_version = jvm.java.lang.System.getProperty("java.version")
scala_version = jvm.scala.util.Properties.versionNumberString()
assert java_version.startswith("17.")
assert scala_version.startswith("2.12.")
assert spark.range(10).agg(F.sum("id")).first()[0] == 45
print(f"PASS: Spark {spark.version}, Java {java_version}, Scala {scala_version}")
print("PASS: local Spark computation (sum 0–9 = 45)")

PASS: Spark 3.5.7, Java 17.0.20.1, Scala 2.12.18
PASS: local Spark computation (sum 0–9 = 45)


### Check Kafka and Structured Streaming
The connectivity check sends one synthetic JSON array through Python and Spark, then removes the temporary topic. Kafka must be running at `127.0.0.1:9092`.

Set `RUN_KAFKA_SMOKE_TEST = False` to skip the check when working offline.

In [3]:
from scripts.check_kafka import check_kafka

if RUN_KAFKA_SMOKE_TEST:
    setup_result = check_kafka(spark, PROJECT_ROOT, KAFKA_BOOTSTRAP_SERVERS)
    print(setup_result)
else:
    print("SKIPPED: Kafka connectivity check")

PASS: Python producer and consumer exchanged one JSON array.


PASS: Spark Structured Streaming parsed the same record using the Kafka connector.
{'python_round_trip': 'PASS', 'spark_streaming': 'PASS', 'records': 1}


## Part A — Machine learning
### A1. Load and validate the approved dataset

### A2. Reserve the streaming partition

### A3. Explore the development data and define the target

### A4. Engineer features and build the pipeline

### A5. Train and compare models

### A6. Save and reload the selected pipeline

## Part B — Real-time streaming
### B1. Kafka producer and message schema

### B2. Read, parse and predict

### B3. Sliding windows, watermark and outputs

### B4. Demonstrate the end-to-end stream

### Reflection

### Stop Spark
Run this cell at the end of the session. Rerun the Spark setup cell before continuing work.

In [4]:
for query in spark.streams.active:
    query.stop()
spark.stop()
print("Spark session stopped.")

Spark session stopped.
